# Lab 7 - Automated Correlation Power Analysis Attack

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the last lab, we showed that there is in fact a linear relationship between Hamming weight and the power consumed by a target by plotting the averages of many traces*

*In this lab, we'll be introducing AES, revisiting a measurement of linearity, then using this information to break AES. This will be done using 
ChipWhisperer Analyzer. After that, we'll analyze the results of the attack and plot some interesting data.*

**LEARNING OUTCOMES:**

* Understanding how correlation can be used to guess AES key bytes
* Understanding why SubBytes is the best location to attack
* Use ChipWhisperer Analyzer to perform a CPA attack
* Plot additional information about the attack

## Prerequisites

Hold up! Before you continue, check you've done the following tutorials:

* ☑ [Lab 6 - Plotting Hamming Weight](../Lab%206%20-%20Plotting%20Hamming%20Weight/Lab.ipynb).

## Intro

In our previous two labs, we showed that there is a linear relationship to the Hamming weight of data being manipulated by our SAM4S and its power consumption. The goal of this lab will be to explain how this information can be used to recover keys from AES, a symmetric encryption algorithm. Then, after that, we will use ChipWhisperer Analyzer, a part of the ChipWhisperer Python library, to recover a key from our SAM4S.

## Setup

Setup and capture are the same as in previous labs:

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
CRYPTO_TARGET = 'TINYAES128C'
SS_VER = 'SS_VER_1_1'

import chipwhisperer as cw
scope = cw.scope()
scope.default_setup() 
target = cw.target(scope, cw.targets.SimpleSerial)    

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 327828                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

In [31]:
%%bash
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=CWHUSKY CRYPTO_TARGET=TINYAES128C SS_VER=SS_VER_1_1 -j

Building for platform CWHUSKY with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
.
Compiling:
-en     simpleserial-aes.c ...
-en     .././simpleserial/simpleserial.c ...
Compiling:
-en     .././hal/hal.c ...
-en     .././hal//sam4s/startup_sam4s.c ...
.
.
Compiling:
.
.
Compiling:
-en     .././hal//sam4s/sam4s_hal.c ...
Compiling:
Compiling:
-en     .././hal//sam4s/uart.c ...
-en     .././hal//sam4s/pio.c ...
-en     .././hal//sam4s/system_sam4s.c ...
.
.
Compiling:
Compiling:
-en     .././hal//sam4s/sysclk.c ...
-en     .././hal//sam4s/pmc.c ...
.
Compilin

In [32]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-CWHUSKY.hex")

## Capturing Traces

In [2]:
from tqdm.notebook import trange
import numpy as np
import time
import shutil, os

path = "lab_plotting_hamming_weight"
if os.path.exists(path):
    shutil.rmtree(path)

project = cw.Project(path)

#print(dir(project))

ktp = cw.ktp.Basic()
trace_array = []
textin_array = []

key, text = ktp.next()

target.set_key(key)

N = 100
for i in trange(N, desc='Capturing traces'):
    scope.arm()
    target.simpleserial_write('p', text)
    
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    
    response = target.simpleserial_read('r', 16)
    
    project.append(cw.TraceContainer(scope.get_last_trace(), text, response, key))
    textin_array.append(text)
    
    key, text = ktp.next() 

Capturing traces:   0%|          | 0/100 [00:00<?, ?it/s]

In [3]:
cw.plot(project.traces[0]) * cw.plot(project.traces[1])

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

## AES

### Quick Summary

AES, the Advanced Encryption Standard, is a symmetric encryption algorithm in widespread use. Notably, it is widely used in both embedded applications to protect communications and firmware, as well as on the internet. In fact, if you open up a typical website and look at the security information for your connection, it's very likely that AES is being used!

In short, the goal of an encryption algorithm is to turn normal data, plaintext, into random looking data, ciphertext, using a secret key. This is called encryption. This ciphertext can later be turned back into plaintext by a related algorithm, turning it back into plaintext in a process known as decryption. For symmetric encryption algorithms like AES, the same key is used for both encryption and decryption. To be considered secure, an attacker should be able to know everything about the encryption system except the key.

To understand how we can use the linear relationship between power and the Hamming weight of data to recover AES keys, it's necessary to learn a bit about how AES works.

AES works on 16 byte blocks, meaning its plaintext, internal state, and ciphertext are all 16 bytes long. To make things simpler, we'll also focus on AES-128, which uses a 16 byte key as well. Note that there are other versions of AES with different key lengths, but analysis of them is similar.

AES's internal state begins as the plaintext and ends as the ciphertext and is organized into a 4x4 matrix:

|  |  |  |  |
| ---- | ---- | ---- | ---- |
| $a_{0,0}$ | $a_{0,1}$ | $a_{0,2}$ | $a_{0,3}$ |
| $a_{1,0}$ | $a_{1,1}$ | $a_{1,2}$ | $a_{1,3}$ |
| $a_{2,0}$ | $a_{2,1}$ | $a_{2,2}$ | $a_{2,3}$ |
| $a_{3,0}$ | $a_{3,1}$ | $a_{3,2}$ | $a_{3,3}$ |

To turn this initial state, the plaintext, into the ciphertext, 4 operations are performed on this state matrix:

* SubBytes
* ShiftRows
* MixColumns
* AddRoundKey

These 4 operations are performed in that order and together are called a round. There are a total of 10 rounds in AES-128, so these operations are performed a total of 10 times with the following exceptions:

1. There is an additional AddRoundKey at the beginning
2. The final round lacks MixColumns

Represented as Python code:

```python
def AES(plaintext, key):
    state = plaintext
    state = AddRoundKey(state, key)
    exp_key = KeyExpansion(key) # different version of key is used each round
    
    for rnd in range(10):
        state = SubBytes(state)
        state = ShiftRows(state)
        if rnd != 9:
            state = MixColumns(state)
        state = AddRoundKey(state, exp_key[rnd])
    ciphertext = state
    return ciphertext
```

Each of these operations is actually fairly simple:

### AES Operations

#### SubBytes

SubBytes puts each byte of the state through a lookup table called the sbox. This table is designed
to remove any linear relationship between the input and output:

```python
def SubBytes(state):
    for r in range(4):
        for c in range(4):
            state[r, c] = sbox[state[r, c]]
    return state
```

<img src="img/sb.png" alt="SPA of Power Analysis h vs 0" width="550"/>

#### ShiftRows

ShiftRows does a rotation of the rows of the columns to the left

```python
def ShiftRows(state):
    sr_lookup = [[1, 2, 3, 0],
                 [2, 3, 0, 1],
                 [3, 0, 1, 2]]
    for row in range(1, 4):
        column in range(4):
            state[row, column] = state[row, sr_lookup[row, column]]

    return state
    
```

<img src="img/sr.png" alt="SPA of Power Analysis h vs 0" width="550"/>

#### MixColumns

MixColumns multiplies each column by a fixed 4x4 matrix. Notably, this means that each
bytes in the output state depends on its columnmates in the input state. That is,
`b[0,0]` depends on `a[n, 0]`

```python
def MixColumns(state):
    for column in range(4):
        state[:, column] = multiply(state[:, column], MC_Matrix)

    return state
```



<img src="img/mc.png" alt="SPA of Power Analysis h vs 0" width="550"/>

XORs the current round key with the state:

```python
def AddRoundKey(state, key):
    for r in range(4):
        for c in range(4):
            state[r, c] = state[r, c] ^ key[r, c]
    return state
```
<img src="img/ark.png" alt="SPA of Power Analysis h vs 0" width="550"/>

### Important Details

There are a few key properties of these operations that I want to highlight:

1. AddRoundKey is what makes the state depend on the key
2. SubBytes makes the algorithm non-linear (important for preventing linear cryptanalysis attacks)
3. MixColumns breaks the independance of bytes. Without it, we would have the same situation as our password check lab where we could guess each byte of the state independent of each other.

Points 1 and 3 mean that, if we could figure out the state between the first AddRoundKey and the first MixColumns, we can trivially recover the key!

Now, we don't know AES's state at this point, but we do have some information about it: recall that our device is leaking the Hamming weight of its internal data. To effectively exploit this, however, we need to learn more about correlation:

## Correlation

We looked at the Pearson Correlation Coefficient a little bit in the previous lab, but as a refresher, it's a measurement of how linear the relationship is
between two datasets. It varies from -1 to 0 to 1, with 0 being non-linear and 1 being perfectly linear. Negative correlations just means the slope of the line is negative. This value is best illustrated by this image:

<img src="img/correlation.png" alt="SPA of Power Analysis h vs 0" width="850"/>

Notably you can see the aforementioned points about the sign being determined by the slope. You can also see that non-linear patterns have a correlation of 0 as well.

Correlation can be calculated as follows:

$r = \frac{cov(X,Y)}{\sigma_x \sigma_y}$

$cov(X,Y) = \frac{1}{N} \sum_{n=1}^{N} [(Y_n - \bar{Y})(X_n - \bar{X})] = \frac{1}{N}((Y-\bar{Y}) \cdot (X - \bar{X}))$

$\sigma_x = \sqrt{\sum_{n=1}^{N}(X_n - \bar{X})^2}$

$\bar{X} = \frac{1}{N}\sum_{n=1}^{N}X_n$

### Using Correlation to Guess Key

So how do we use this to recover the key? Well, let's capture a bunch of traces and make our two datasets for the correlation calculation a guess at the leakage for an AES operation and our power trace. Remember that the leakage is just the Hamming weight of the data being manipulated. So, if we pick our operation to be the output of AddRoundKey, we end up with `X = HW[pt ^ key]` and `Y=trace[op_time]`. Considering `pt` is known and if we know `op_time`, this leads to the following situation:

1. If all the guesses at the leakage are correct (we successfully guess `key`), we should get a high correlation between `X` and `Y`
2. If most of the guesses at the leakage are incorrect (our guess of `key` is wrong), we should get a low correaltion between `X` and `Y`.

This means we can use correlation to guess the correct value of `key`! Actually, we can also use correlation to figure out `op_time` as well, as incorrect values for this should also give a low correlation.

Unfortunately, things aren't quite that simple - if we try this, we'll find that some incorrect guesses also give a high correlation. This is because the XOR used in Add Round Key is a **linear operation**. To better understand why this is an issue, let's try plotting the "real" leakage (aka `HW[pt ^ key]`) against some guesses for the leakage:

## Attack Location

### AddRoundKey



In [5]:
import chipwhisperer as cw
import numpy as np
def generate_hw_table():
    ret = []
    for i in range(256):
        ret.append(bin(i).count('1'))
    return np.array(ret, dtype=np.uint8)

hw_table = generate_hw_table()

In [7]:
print(hw_table)

[0 1 1 2 1 2 2 3 1 2 2 3 2 3 3 4 1 2 2 3 2 3 3 4 2 3 3 4 3 4 4 5 1 2 2 3 2
 3 3 4 2 3 3 4 3 4 4 5 2 3 3 4 3 4 4 5 3 4 4 5 4 5 5 6 1 2 2 3 2 3 3 4 2 3
 3 4 3 4 4 5 2 3 3 4 3 4 4 5 3 4 4 5 4 5 5 6 2 3 3 4 3 4 4 5 3 4 4 5 4 5 5
 6 3 4 4 5 4 5 5 6 4 5 5 6 5 6 6 7 1 2 2 3 2 3 3 4 2 3 3 4 3 4 4 5 2 3 3 4
 3 4 4 5 3 4 4 5 4 5 5 6 2 3 3 4 3 4 4 5 3 4 4 5 4 5 5 6 3 4 4 5 4 5 5 6 4
 5 5 6 5 6 6 7 2 3 3 4 3 4 4 5 3 4 4 5 4 5 5 6 3 4 4 5 4 5 5 6 4 5 5 6 5 6
 6 7 3 4 4 5 4 5 5 6 4 5 5 6 5 6 6 7 4 5 5 6 5 6 6 7 5 6 6 7 6 7 7 8]


Use the above code to **plot the actual leaked information against a guess of the leaked information**. You may find it helpful to also **plot the leaked information against itself**:

In [6]:
list = [[] for _ in range(9)]
mean = []
for i, tra in enumerate(project.traces):
    ch = project.plaintexts[i][0]
    hw = hw_table[ch]
    if len(tra) != 0:
        list[hw].append(tra)
        #print(project.plaintexts[i])
        #print(ch, " ", hw)


for i in range(9):
    #print(len(list[i]))
    if len(list[i]) != 0:
        mean.append( np.mean(list[i], axis=0))
    else:
        mean.append(None)
    
#print(mean[0])
#print(mean[2])
import holoviews as hv

print(key)

plot = cw.plot([])
for mean_trace in mean:
    if mean_trace is not None:
        plot *= cw.plot(mean_trace)[520:550].opts(color=hv.Palette('Spectral'))
plot

CWbytearray(b'2b 7e 15 16 28 ae d2 a6 ab f7 15 88 09 cf 4f 3c')


:Overlay
   .Curve.I    :Curve   [x]   (y)
   .Curve.II   :Curve   [x]   (y)
   .Curve.III  :Curve   [x]   (y)
   .Curve.IV   :Curve   [x]   (y)
   .Curve.V    :Curve   [x]   (y)
   .Curve.VI   :Curve   [x]   (y)
   .Curve.VII  :Curve   [x]   (y)
   .Curve.VIII :Curve   [x]   (y)
   .Curve.IX   :Curve   [x]   (y)

I recommend trying a few values here, especially the **inverse of the key**.

So should we give up on our attack? The answer is no, all we need to do is introduce a non-linear operation into our leakage function. Recall
that the very next operation in AES, **SubBytes**, is specifically designed to be non-linear. Let's try repeating our plot, replacing our ARK leakages with
SubBytes instead:

### SubBytes

In [7]:
sbox = np.array([
# 0    1    2    3    4    5    6    7    8    9    a    b    c    d    e    f 
0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76, # 0
0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0, # 1
0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15, # 2
0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75, # 3
0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84, # 4
0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf, # 5
0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8, # 6
0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2, # 7
0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73, # 8
0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb, # 9
0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79, # a
0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08, # b
0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a, # c
0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e, # d
0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf, # e
0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16  # f
], dtype=np.uint8)

**Repeat your plots from above, but change the leakage to be from the SubBytes output instead of the AddRoundKey output:**

In [15]:
list = [[] for _ in range(9)]
mean = []
key_guess = 0x2b
for i, tra in enumerate(project.traces):
    ch = project.plaintexts[i][0]
    hw = hw_table[sbox[ch ^ key_guess]]
    if len(tra) != 0:
        list[hw].append(tra)
        #print(project.plaintexts[i])
        #print(ch, " ", hw)


for i in range(9):
    #print(len(list[i]))
    if len(list[i]) != 0:
        mean.append( np.mean(list[i], axis=0))
    else:
        mean.append(None)
    
#print(mean[0])
#print(mean[2])
import holoviews as hv

plot = cw.plot([])
for mean_trace in mean:
    if mean_trace is not None:
        plot *= cw.plot(mean_trace).opts(color=hv.Palette('Spectral'))
plot

:Overlay
   .Curve.I    :Curve   [x]   (y)
   .Curve.II   :Curve   [x]   (y)
   .Curve.III  :Curve   [x]   (y)
   .Curve.IV   :Curve   [x]   (y)
   .Curve.V    :Curve   [x]   (y)
   .Curve.VI   :Curve   [x]   (y)
   .Curve.VII  :Curve   [x]   (y)
   .Curve.VIII :Curve   [x]   (y)

Looks like we've solved the issue! So now we have `X = HW[sbox[pt ^ key_guess]]` and `Y=trace[op_guess]`. As a rough approximation, our attack will for 1 byte will look like:

```python
best_t = 0
best_guess = 0
best_corr = 0
for t in range(project.trace_len):
    for guess in range(256):
        r = correlation(project.traces[:,t], leakage(pt[:,0], guess))
        if r > best_corr:
            best_corr = r
            best_t = t
            best_guess = guess
```

Even for this small attack, we have a lot of data points, which makes this calculation a little tricky to do in a reasonable amount of time. This is why we'll be using a submodule in ChipWhisperer optimized for these calculations. This submodule is called ChipWhisperer Analyzer.

## Running the Attack

Not only is Analyzer optimized for correlation calculations, it's also designed to fit in nicely with our ChipWhisperer Projects. There are four things
you need to do to run a CPA attack with Analyzer:

1. Create a leakage model. Common ones are available from `chipwhisperer.analyzer.leakage_models`, including the sub bytes output (as `leakage_models.sbox_output`)
2. Create the CPA object, passing your project, leakage model, and the bytes you want to attack
3. Run the attack
4. Interpret the results

You'll also probably want to create a callback function to get updates as the attack runs. We've got a nice table you can use available via `cb = get_table_cb()`, also available from the analyzer submodule.

**Create the `CPA` object, pass it your `project` and the `leakage_models.sbox_output` model, and `run()` it against all 16 bytes:**

In [17]:
import numpy as np
import chipwhisperer.analyzer as cwa
del list            # restores the built-in list()

leak_model = cwa.leakage_models.sbox_output
attack = cwa.CPA(project, leak_model, subkeys=list(range(16)))
attack.run()                      # returns None in 6.1.0

# do NOT print(attack) – it's broken
for name, val in vars(attack).items():
    info = getattr(val, "shape", None) or type(val).__name__
    print(f"{name:25s} {info}")
print([n for n in dir(attack) if not n.startswith("_")])

project                   Project
trace_len                 int
kguesses                  int
traces_used               int
subkeys                   list
num_traces                int
trace_array               (100, 5000)
pt_array                  (16, 100)
ct_array                  (16, 100)
th_sum                    (16, 5000, 256)
tsum                      (5000,)
hsum                      (16, 256)
t2_sum                    (5000,)
h2_sum                    (16, 256)
hyp_array                 (16, 100, 256)
sorted_kguesses_hist      list
traces_used_hist          list
max_correlations_hist     list
known_key                 (16,)
correlations              (16, 5000, 256)
leakage_model             function
best_corrs                (16, 256)
max_corr_loc              (256,)
['avg_pge', 'best_corrs', 'calculate_correlation', 'corr_v_time', 'corr_v_time_plot', 'corr_v_traces', 'corr_v_traces_plot', 'correlations', 'ct_array', 'gen_hyp', 'h2_sum', 'highest_corr_v_time', 'highest_corr_

In [20]:
import chipwhisperer.analyzer as cwa
leak_model = cwa.leakage_models.sbox_output
attack = cwa.CPA(project, leak_model, subkeys=list(range(16)))
results = attack.run()
#del list            # restores the built-in list()

import numpy as np

best = np.abs(attack.best_corrs)                     # (16, 256)
key_guess = np.argmax(best, axis=1).astype(np.uint8)
real_key = np.array(attack.known_key, dtype=np.uint8)

print("Recovered:", bytes(key_guess).hex())
print("Real key :", bytes(real_key).hex())
print("Correct bytes:", int(np.sum(key_guess == real_key)), "/ 16")

Recovered: 2b7e151628aed2a6abf7158809cf4f3c
Real key : 2b7e151628aed2a6abf7158809cf4f3c
Correct bytes: 16 / 16


In [19]:
from IPython.display import clear_output, display
import numpy as np
import pandas as pd
import chipwhisperer.analyzer as cwa
#del list            # restores the built-in list()

def format_stat(stat):
    return "{:02X}<br>{:.3f}".format(int(stat[0]), float(stat[2]))

def color_corr_key(row):
    ret = [""] * len(row)
    for i, stat in enumerate(row):
        if int(stat[0]) == int(key[i]):
            ret[i] = "color: red"
    return ret

def find_maximums(atk):
    """6.1.0 replacement for results.find_maximums():
       per byte a list of (kguess, location, corr), sorted by corr."""
    best = np.abs(atk.best_corrs)                      # (16, 256)
    corrs = np.abs(atk.correlations)                   # (16, samples, 256)
    stat_data = []
    for b in range(best.shape[0]):
        order = np.argsort(-best[b])
        stat_data.append([(int(kg),
                           int(np.argmax(corrs[b, :, kg])),
                           float(best[b, kg])) for kg in order])
    return stat_data

def stats_callback(atk):
    stat_data = find_maximums(atk)
    df = pd.DataFrame(stat_data).transpose()           # rows = rank, cols = byte
    clear_output(wait=True)
    display(df.head().style.format(format_stat).apply(color_corr_key, axis=1))

leak_model = cwa.leakage_models.sbox_output
attack = cwa.CPA(project, leak_model, subkeys=list(range(16)))
attack.run(interval=10, callback=stats_callback)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,2B0.824,7E0.862,150.780,160.821,280.758,AE0.765,D20.808,A60.804,AB0.653,F70.879,150.842,880.821,090.652,CF0.766,4F0.804,3C0.784
1,440.473,AC0.457,3B0.471,A70.462,240.480,460.471,EE0.530,4C0.481,4E0.525,5C0.486,950.449,F80.490,610.503,900.507,9E0.496,C40.467
2,2A0.462,B90.453,B60.464,700.458,5E0.467,910.468,130.508,8D0.470,970.464,5E0.484,850.446,2C0.473,CD0.467,A30.483,CA0.492,8F0.459
3,E90.450,1E0.450,930.462,FD0.444,0B0.454,050.442,A70.460,300.469,DE0.462,C50.452,430.439,C00.471,690.461,7F0.462,E70.458,700.456
4,A30.448,DA0.448,640.444,900.439,2A0.452,D00.439,160.452,D60.453,5A0.456,B10.451,B00.436,9D0.467,960.461,860.459,6B0.456,EF0.451


## Results

You should have pretty quickly seen the correct key (which you can see via `project.keys[0]`) filter to the top. The correct key is also in red text, which should make it pretty obvious if you've recovered the correct key. We've also got some nice convenience functions to print the key guess (`cpa.key_guess()`) and whether or not you've got the correct key (`cpa.key_recovered()`).

**Do that below:**

Of course, these methods of identifying whether or not you've recovered the key only work if you know the key, making them useful for evaluating targets, but not for actual attacks.

For actual attacks, you may be able to make a reasonable guess at the ciphertext, though this is more likely in the the case of attacking decryption rather than encryption. Another method you can use is to look at the difference in correlation between the best guess and the next guesses. If your best guess has a much higher correlation, it's very likely that it's the correct key. You should see this in the attack we just ran!

If you want, you can also pass the CPA object to the callback function to get the table back out.

## Analyzer Plots

Before we end this lab, I want to go through some of the other interesting things we can learn from this attack. The first is when in time our SubBytes operation is happening. Remember that we perform the correlation with all key guesses at all points in time for our power trace and that we should only see a high correlation when our SubBytes operation is happening. This means that, if we plot correlation for our key guess, we should see a spike at a particular time.

Analyzer has a convenient method, `cpa.corr_v_time_plot()` for doing just that:

### Corr v. Time

### PGE v. Traces

There's two more plots that Analyzer can do. Both are important for understanding how many traces an attack requires. The first of these two plots is Partial Guessing Entropy (PGE) vs. Number of Traces. This sounds complicated, but PGE is just the number of incorrect key bytes ranked above the correct key guess. For example, if the correct value for key byte 0 is 0xAD and 0x20, 0x45, and 0x91 have higher correlations, then the PGE for that byte is 3.

By looking at when the PGE for all bytes reaches 0, you can determine the minimum number of traces you needed for your attack. This can be a very useful metric for evaluating target devices.

**Do a `cpa.pge_v_traces_plot()` below:**

### Corr v. Traces

The final plot we'll look at is correlation vs trace used. This is similar to PGE vs. Traces, but this one is more useful for determining when the correct key guesses are distinct from the other guesses. This is very useful for understanding when an attacker might know when they've recovered the key.

**Do a `cpa.corr_v_traces_plot()` below:**

## Conclusions & Next Steps

Congratulations, you've successfully recovered your first AES key! Analyzer makes this attack easy and has the advantage of capturing some interesting data for us.

In the next lab, we're going to perform this same attack without using Analyzer. 

**We're going to use the location of the SubBytes operation from the correlation vs. time plot to narrow the attack down to a single point. Make sure you record this value.**

In [29]:
scope.dis()
target.dis()

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>